In [2]:
import glob
import os

import pandas as pd
import numpy as np

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

import skimage
from skimage.util import compare_images
from skimage import io, color, measure
from skimage.measure import regionprops
from skimage.measure import label as sk_label  # Renaming the label function to avoid conflicts

import matplotlib.pyplot as plt
import matplotlib

%load_ext autoreload
%autoreload 2
%matplotlib inline


In [18]:
# Import images

merged_dir = '/Volumes/bamfaile/Analysis/TUBB2B-KI/smFISH_organoids/D56/20240925_TUBB2B-KI_MCP-Halo-JF503_TUBB2B-565_MS2-633/Regular/Max_projections'
merged_path = os.path.join(merged_dir,'*.tif')
merged_files = glob.glob(merged_path)

In [21]:
# Read images into list

merged_images = []

for file in merged_files:
    image = imread(file)
    merged_images.append(image)

In [ ]:
for i, image in enumerate(merged_images):
    print(f"Image {i + 1}: Shape = {image.shape}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(merged_images[0][0])
ax[1].imshow(merged_images[0][1])

In [ ]:
# Flip first channel

flipped_images = []

for image in merged_images:
    # Flip the first channel by 90 degrees
    flipped_channel1 = np.rot90(image[0])  # Rotate the first channel 90 degrees counter-clockwise
    channel2 = image[1]  # Second channel remains unchanged
    
    # Stack the modified channels back
    modified_image = np.stack([flipped_channel1, channel2], axis=0)
    flipped_images.append(modified_image)

print(len(flipped_images))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(flipped_images[0][0])
ax[1].imshow(flipped_images[0][1])

In [26]:
# Save cytoplasm masks

output_dir = '/Volumes/bamfaile/Analysis/TUBB2B-KI/smFISH_organoids/D56/20240925_TUBB2B-KI_MCP-Halo-JF503_TUBB2B-565_MS2-633/Flipped/Max_projections'


In [ ]:
for img, tiff_file in zip(flipped_images, merged_files):
    
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]
    print(file_name)
    
    # Define the output file path
    output_file = os.path.join(output_dir, file_name + '.tif')
    print(output_file)
    
    # Save the manipulated image as a TIFF
    tf.imwrite(output_file, img)